In [ ]:
!pip -q install -U transformers accelerate bitsandbytes sentencepiece scipy
!pip -q install -U fastapi uvicorn pydantic

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.3/62.3 kB 2.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 15.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.3/35.3 MB 14.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 130.2/130.2 kB 7.3 MB/s eta 0:00:00


In [ ]:
from transformers import AutoTokenizer

MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# Tokenizer will be loaded as part of the pipeline or AutoModelForSequenceClassification
# No separate tokenizer initialization here as it's not strictly necessary for this cell's output
print("Model name updated for zero-shot classification!")

Model name updated for zero-shot classification!


In [ ]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

# For zero-shot classification with DeBERTa-v3-base, 4-bit quantization is generally not needed
# and might cause issues or not be fully supported by the pipeline directly.
# We will use the model directly with the pipeline.

# Load tokenizer and model for the zero-shot classification pipeline
tokenizer = AutoTokenizer.from_pretrained(MODEL)
model = AutoModelForSequenceClassification.from_pretrained(MODEL)

print("✅ Zero-shot Classification Model Loaded!")

config.json:   0%|          | 0.00/1.09k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/1.28k [00:00<?, ?B/s]

spm.model: reconstructing file:   0%|          |  0.00B / 2.46MB            

spm.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/8.66M [00:00<?, ?B/s]

...

special_tokens_map.json:   0%|          | 0.00/286 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  369MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/202 [00:00<?, ?it/s]

✅ Zero-shot Classification Model Loaded!


In [ ]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel
from typing import List
from transformers import pipeline

MODEL = "MoritzLaurer/DeBERTa-v3-base-mnli-fever-anli"

print("Loading Zero-Shot Classification Model...")

classifier = pipeline(
    "zero-shot-classification",
    model=MODEL
)

print("Model Loaded Successfully!")

app = FastAPI()


class ChatRequest(BaseModel):
    text: str
    candidate_labels: List[str]


@app.get("/")
def home():
    return {
        "status": "running",
        "model": MODEL
    }


@app.get("/health")
def health():
    return {
        "status": "healthy"
    }


@app.post("/chat")
def chat(req: ChatRequest):

    result = classifier(
        req.text,
        candidate_labels=req.candidate_labels,
        multi_label=False
    )

    return {
        "success": True,
        "label": result["labels"][0],
        "score": float(result["scores"][0]),
        "labels": result["labels"],
        "scores": [float(x) for x in result["scores"]]
    }


@app.post("/classify")
def classify(req: ChatRequest):
    return chat(req)

Writing app.py


In [ ]:
!nohup uvicorn app:app --host 0.0.0.0 --port 8000 > server.log 2>&1 &

In [ ]:
!ps -ef | grep uvicorn

root        1375       1  0 13:55 ?        00:00:00 /usr/bin/python3 /usr/local/bin/uvicorn app:app --host 0.0.0.0 --port 8000
root        1376     822  0 13:55 ?        00:00:00 /bin/bash -c ps -ef | grep uvicorn
root        1378    1376  0 13:55 ?        00:00:00 grep uvicorn


In [ ]:
!tail -50 server.log

Loading Zero-Shot Classification Model...
Loading weights: 100%|██████████| 202/202 [00:00<00:00, 13098.49it/s]
INFO:     Started server process [1375]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


In [ ]:
!curl http://127.0.0.1:8000/health

{"status":"healthy"}

In [ ]:
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8000

2026-07-21T13:56:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-21T13:56:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-21T13:56:07Z INF +--------------------------------------------------------------------------------------------+
2026-07-21T13:56:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-21T13:56:07Z INF |  https://telephone-surge-boundaries-oscar.trycloudflar

The `cloudflared` output above provides a public URL for your FastAPI application. Let's extract that URL and use it to interact with the API.

In [ ]:
import re

# Extract the URL from the cloudflared output
cloudflared_output = """
2026-07-21T13:56:04Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-07-21T13:56:04Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-07-21T13:56:07Z INF +--------------------------------------------------------------------------------------------+
2026-07-21T13:56:07Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-07-21T13:56:07Z INF |  https://telephone-surge-boundaries-oscar.trycloudflare.com                                |
2026-07-21T13:56:07Z INF +--------------------------------------------------------------------------------------------+
2026-07-21T13:56:07Z INF Cannot determine default configuration path. No file [config.yml config.yaml] in [~/.cloudflared ~/.cloudflare-warp ~/cloudflare-warp /etc/cloudflared /usr/local/etc/cloudflared]
2026-07-21T13:56:07Z INF Version 2026.7.2 (Checksum ec905ea7b7e327ff8abdde8cb64697a2152de74dbcdbf6aec9db8364eb3886cd)
2026-07-21T13:56:07Z INF GOOS: linux, GOVersion: go1.26.4, GoArch: amd64
2026-07-21T13:56:07Z INF Settings: map[ha-connections:1 protocol:quic url:http://127.0.0.1:8000]
2026-07-21T13:56:07Z INF cloudflared will not automatically update when run from the shell. To enable auto-updates, run cloudflared as a service: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps/configure-tunnels/local-management/as-a-service/
"

match = re.search(r'https://[a-zA-Z0-9.-]+\.trycloudflare\.com', cloudflared_output)
if match:
    tunnel_url = match.group(0)
    print(f"Cloudflare Tunnel URL: {tunnel_url}")
else:
    tunnel_url = None
    print("Could not find Cloudflare Tunnel URL in the output.")


Now, let's use the extracted `tunnel_url` to send a sample request to the `/chat` endpoint of your FastAPI application.

In [ ]:
import requests
import json

if tunnel_url:
    chat_endpoint = f"{tunnel_url}/chat"

    # Example payload for the /chat endpoint
    chat_payload = {
        "text": "I love this movie!",
        "candidate_labels": ["positive", "negative", "neutral"]
    }

    try:
        response = requests.post(chat_endpoint, json=chat_payload)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        chat_result = response.json()
        print("\nResponse from /chat endpoint:")
        print(json.dumps(chat_result, indent=2))

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to /chat endpoint: {e}")
else:
    print("Cannot test /chat endpoint without a tunnel URL.")


You can also use the `/classify` endpoint, which is aliased to `/chat` in your `app.py` file. Here's an example:

In [ ]:
if tunnel_url:
    classify_endpoint = f"{tunnel_url}/classify"

    # Example payload for the /classify endpoint
    classify_payload = {
        "text": "This product is terrible, I want my money back.",
        "candidate_labels": ["electronics", "food", "customer service", "returns"]
    }

    try:
        response = requests.post(classify_endpoint, json=classify_payload)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        classify_result = response.json()
        print("\nResponse from /classify endpoint:")
        print(json.dumps(classify_result, indent=2))

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to /classify endpoint: {e}")
else:
    print("Cannot test /classify endpoint without a tunnel URL.")


Here's the Python code for interacting with the `/chat` endpoint. Save this as a `.py` file (e.g., `test_chat.py`) and run it from your terminal.

In [ ]:
import requests
import json

# --- IMPORTANT: Replace with your actual Cloudflare Tunnel URL ---
tunnel_url = "YOUR_CLOUDFLARE_TUNNEL_URL"
# Example: tunnel_url = "https://telephone-surge-boundaries-oscar.trycloudflare.com"
# ----------------------------------------------------------------

if tunnel_url == "YOUR_CLOUDFLARE_TUNNEL_URL":
    print("Please replace 'YOUR_CLOUDFLARE_TUNNEL_URL' with your actual Cloudflare Tunnel URL.")
else:
    chat_endpoint = f"{tunnel_url}/chat"

    # Example payload for the /chat endpoint
    chat_payload = {
        "text": "I love this movie!",
        "candidate_labels": ["positive", "negative", "neutral"]
    }

    try:
        response = requests.post(chat_endpoint, json=chat_payload)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        chat_result = response.json()
        print("\nResponse from /chat endpoint:")
        print(json.dumps(chat_result, indent=2))

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to /chat endpoint: {e}")


And here's the Python code for interacting with the `/classify` endpoint. Save this as a separate `.py` file (e.g., `test_classify.py`) and run it from your terminal.

In [ ]:
import requests
import json

# --- IMPORTANT: Replace with your actual Cloudflare Tunnel URL ---
tunnel_url = "YOUR_CLOUDFLARE_TUNNEL_URL"
# Example: tunnel_url = "https://telephone-surge-boundaries-oscar.trycloudflare.com"
# ----------------------------------------------------------------

if tunnel_url == "YOUR_CLOUDFLARE_TUNNEL_URL":
    print("Please replace 'YOUR_CLOUDFLARE_TUNNEL_URL' with your actual Cloudflare Tunnel URL.")
else:
    classify_endpoint = f"{tunnel_url}/classify"

    # Example payload for the /classify endpoint
    classify_payload = {
        "text": "This product is terrible, I want my money back.",
        "candidate_labels": ["electronics", "food", "customer service", "returns"]
    }

    try:
        response = requests.post(classify_endpoint, json=classify_payload)
        response.raise_for_status()  # Raise an exception for HTTP errors (4xx or 5xx)
        classify_result = response.json()
        print("\nResponse from /classify endpoint:")
        print(json.dumps(classify_result, indent=2))

    except requests.exceptions.RequestException as e:
        print(f"Error connecting to /classify endpoint: {e}")
